Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
CC3084 - Data Science - sección 40

Cristian Túnchez (231359)  
Nadissa Vela (23764)

# Laboratorio 6: Análisis de Redes Sociales en YouTube

## Notebook 1 - Carga, comprensión, integración y limpieza

Este notebook resuelve los **Ejercicios 1 y 2** del laboratorio.

En este notebook se cargan los dos conjuntos de datos recolectados de YouTube, se identifica cómo se relacionan entre sí, se diagnostica su calidad y se produce una versión limpia de ambos que servirá de insumo para el análisis exploratorio y para la construcción de las redes.

**Preparación del entorno**

Se importan las librerías necesarias y se fija un estilo común para todos los gráficos. El módulo `src/utils.py` aporta únicamente las rutas del proyecto y ayudas de guardado.

In [1]:
import re
import sys
import ast
import unicodedata
from urllib.parse import unquote
from pathlib import Path

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import DATA_RAW, set_estilo, guardar

set_estilo()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("pandas:", pd.__version__, "| spaCy:", spacy.__version__)
print("Datos crudos en:", DATA_RAW)

pandas: 2.3.3 | spaCy: 3.8.16
Datos crudos en: /home/sebas/github/Data-Science/CC3084-Laboratorio-6/data/raw


## Ejercicio 1. Carga, comprensión e integración de los datos

### 1.1 Carga de los archivos

In [2]:
videos = pd.read_csv(DATA_RAW / "youtube_videos.csv")
comentarios = pd.read_csv(DATA_RAW / "youtube_comments.csv")

print(f"youtube_videos.csv   -> {videos.shape[0]} filas x {videos.shape[1]} columnas")
print(f"youtube_comments.csv -> {comentarios.shape[0]} filas x {comentarios.shape[1]} columnas")

youtube_videos.csv   -> 293 filas x 20 columnas
youtube_comments.csv -> 406 filas x 17 columnas


In [3]:
# Primeras filas de cada archivo, transpuestas para poder leer todas las columnas
videos.head(2).T

,0,1
video_id,-5puKGEqcUc,-E7OPOLjMug
title,INSIVUMEH pronostica incremento de lluvias par...,BERNARDO ARÉVALO CALIFICA CAMBIO EN EL MP COMO...
channel_name,T13 Noticias Guatemala,IDocumenta
channel_id,UCq0Cm-3SKthEySQc2JZBi1A,UCgItjn_ZWFWcv1MlpKIkqgQ
source_query,guatemala lluvias,@GobiernodelaRepublicadeGuatema
source_group,topic,topic
dataset_sources,youtube_guatemala.csv | youtube_guatemala_lab....,youtube_target_channels.csv
channel_handle,/@T13NoticiasGuatemala,/@iDocumenta
published_time,hace 2 días,hace 3 meses
view_count_text,"2,390 vistas",4 vistas


In [4]:
comentarios.head(2).T

,0,1
video_id,j43HgwYFKfk,06mFNPU0aB8
comment_id,Ugw-J65a1iYL9hqhELh4AaABAg,Ugw-ZT9t9wU2V-tCaUZ4AaABAg
video_title,La cooptación de Walter Mazariegos en la USAC,Capturan a presuntos delincuentes disfrazados ...
channel_name,Quorum,Noti7
channel_id,UCE4rsXcgDb6e1-a9iTbWzfg,UCVpSRoZgngfSL03Nlbjtq9A
author_name,@MarcosCarillo-b1r,@RaulPerez-cw2vi
author_channel_id,UCdFlugHJJa4l3YqWuNRmvXw,UCvl1tzQeBeGy6efPTRJXSCw
text,Ese corrupto amigo de la vieja fiscal los teng...,"Están jóvenes porque no buscan un trabajo, tu..."
source_query,@quorumgt,guatemala noticias
source_group,topic,topic


### 1.2 Unidad de observación, llave primaria y variables relevantes

Se comprueba con los datos que las llaves primarias son efectivamente únicas y no tienen valores faltantes.

In [5]:
def revisar_llave(df, columna, nombre):
    return {
        "conjunto": nombre,
        "llave": columna,
        "filas": len(df),
        "valores_unicos": df[columna].nunique(),
        "es_unica": bool(df[columna].is_unique),
        "faltantes": int(df[columna].isna().sum()),
    }

pd.DataFrame([
    revisar_llave(videos, "video_id", "youtube_videos"),
    revisar_llave(comentarios, "comment_id", "youtube_comments"),
])

,conjunto,llave,filas,valores_unicos,es_unica,faltantes
0,youtube_videos,video_id,293,293,True,0
1,youtube_comments,comment_id,406,406,True,0


In [6]:
# Clasificación de las variables de cada archivo por su papel en el análisis
familias = {
    "youtube_videos": {
        "Identificadores": ["video_id", "channel_id", "channel_handle", "owner_handle", "video_url"],
        "Contenido / texto": ["title", "description", "description_snippet", "keywords"],
        "Conteos": ["view_count", "view_count_text"],
        "Temporales": ["publish_date", "upload_date", "published_time"],
        "Clasificación": ["category", "channel_name"],
        "Procedencia del muestreo": ["source_query", "source_group", "query_hits", "dataset_sources"],
    },
    "youtube_comments": {
        "Identificadores": ["comment_id", "video_id", "author_channel_id", "channel_id", "author_handle"],
        "Contenido / texto": ["text"],
        "Conteos": ["like_count_text", "reply_count"],
        "Temporales": ["published_text"],
        "Clasificación": ["author_name", "channel_name", "video_title", "is_pinned", "viewer_rating"],
        "Procedencia del muestreo": ["source_query", "source_group", "dataset_sources"],
    },
}

filas = [
    {"conjunto": conj, "familia": fam, "n": len(cols), "variables": ", ".join(cols)}
    for conj, grupos in familias.items()
    for fam, cols in grupos.items()
]
pd.DataFrame(filas)

,conjunto,familia,n,variables
0,youtube_videos,Identificadores,5,"video_id, channel_id, channel_handle, owner_ha..."
1,youtube_videos,Contenido / texto,4,"title, description, description_snippet, keywords"
2,youtube_videos,Conteos,2,"view_count, view_count_text"
3,youtube_videos,Temporales,3,"publish_date, upload_date, published_time"
4,youtube_videos,Clasificación,2,"category, channel_name"
5,youtube_videos,Procedencia del muestreo,4,"source_query, source_group, query_hits, datase..."
6,youtube_comments,Identificadores,5,"comment_id, video_id, author_channel_id, chann..."
7,youtube_comments,Contenido / texto,1,text
8,youtube_comments,Conteos,2,"like_count_text, reply_count"
9,youtube_comments,Temporales,1,published_text


**Unidad de observación y llave primaria**

En `youtube_videos` cada fila es un **video** y la llave primaria es `video_id`; en `youtube_comments` cada fila es un **comentario principal** y la llave primaria es `comment_id`. Ambas llaves resultaron únicas y sin valores faltantes, por lo que pueden usarse con confianza para relacionar los archivos.

**Variables relevantes**

Se agruparon en seis grupos según su rol:

- Los **identificadores** (`video_id`, `channel_id`, `comment_id`, `author_channel_id`) son los únicos que se usarán como llaves y como nodos de las redes.
- El **contenido textual** (`title`, `description`, `text`) alimenta el análisis de palabras, temas y sentimiento.
- Los **conteos** (`view_count`, `like_count_text`, `reply_count`) miden popularidad y reacción.
- Las variables **temporales** de este conjunto son relativas al momento de la recolección y por eso su uso es limitado.
- La **clasificación** (`category`, nombres visibles) describe el contenido.
- La **procedencia del muestreo** (`source_query`, `source_group`, `query_hits`) describe cómo se encontró cada registro, no de qué trata.

### 1.3 Relación entre canal, video, autor del comentario, comentario, categoría y consulta

Se cuantifica cada entidad y se verifica si el espacio de identificadores de los **canales que publican videos** se traslapa con el de los **autores de comentarios**.

In [7]:
resumen_entidades = pd.Series({
    "canales que publican videos (channel_id)": videos["channel_id"].nunique(),
    "videos (video_id)": videos["video_id"].nunique(),
    "comentarios (comment_id)": comentarios["comment_id"].nunique(),
    "autores de comentarios (author_channel_id)": comentarios["author_channel_id"].nunique(),
    "categorías de video (category)": videos["category"].nunique(),
    "consultas de búsqueda en videos (source_query)": videos["source_query"].nunique(),
    "consultas de búsqueda en comentarios (source_query)": comentarios["source_query"].nunique(),
}, name="valores únicos")
resumen_entidades.to_frame()

,valores únicos
canales que publican videos (channel_id),97
videos (video_id),293
comentarios (comment_id),406
autores de comentarios (author_channel_id),332
categorías de video (category),11
consultas de búsqueda en videos (source_query),21
consultas de búsqueda en comentarios (source_query),6


In [8]:
canales_dueños = set(videos["channel_id"])
autores = set(comentarios["author_channel_id"])

print("Autores que además son dueños de algún video del conjunto:", len(canales_dueños & autores))
print()
print("channel_id dentro de youtube_comments identifica al DUEÑO del video comentado.")
print("¿Coincide siempre con el channel_id del video en youtube_videos?")
verif = comentarios[["video_id", "channel_id"]].merge(
    videos[["video_id", "channel_id"]], on="video_id", suffixes=("_comentario", "_video")
)
print("  coincidencias:", (verif["channel_id_comentario"] == verif["channel_id_video"]).sum(), "de", len(verif))

Autores que además son dueños de algún video del conjunto: 0

channel_id dentro de youtube_comments identifica al DUEÑO del video comentado.
¿Coincide siempre con el channel_id del video en youtube_videos?
  coincidencias: 406 de 406


In [9]:
# Un video pertenece a una categoría y fue recuperado por una o varias consultas
print("Distribución de source_group en videos:")
print(videos["source_group"].value_counts().to_string())
print("\nDistribución de source_group en comentarios:")
print(comentarios["source_group"].value_counts().to_string())
print("\nVideos recuperados por más de una consulta (query_hits):",
      sum(len(ast.literal_eval(x)) > 1 for x in videos["query_hits"]))

Distribución de source_group en videos:
source_group
topic           177
official_gov    105
channel          11

Distribución de source_group en comentarios:
source_group
channel    231
topic      175

Videos recuperados por más de una consulta (query_hits): 5


**¿Cómo se relacionan las entidades?**

La estructura observada es la siguiente:

```
canal (channel_id) ──publica──> video (video_id) ──recibe──> comentario (comment_id)
                                     │                              │
                              category (1 por video)        autor (author_channel_id)
                                     │
                     source_query / source_group / query_hits  (cómo se recolectó)
```

- Un **canal** publica uno o varios videos; en este conjunto hay 97 canales para 293 videos.
- Un **video** pertenece a exactamente una **categoría** y recibe cero o varios comentarios.
- Un **comentario** pertenece a un solo video y tiene un solo **autor**.
- Un **autor** puede comentar en varios videos.

**Verificaciones importantes**

1. **Los autores y los canales dueños son conjuntos separados.** Ninguno de los 332 autores de comentarios es dueño de alguno de los 293 videos. Por eso `channel_id` y `author_channel_id` no deben mezclarse, aunque ambos son identificadores de canal de YouTube, en este conjunto describen roles distintos.
2. **`channel_id` dentro de `youtube_comments` identifica al dueño del video comentado**, no al autor del comentario, y coincide con el del catálogo de videos en los 406 casos.

Finalmente, `source_query` y `source_group` describen **el procedimiento de recolección**. 177 videos se hallaron por tema, 105 por canales oficiales de gobierno y 11 por canal. Cinco videos fueron recuperados por más de una consulta. Que un video aparezca bajo la consulta *guatemala lluvias* no garantiza que trate sobre lluvias.

### 1.4 Integración de los conjuntos mediante `video_id`

In [10]:
comentarios_con_video = comentarios["video_id"].isin(videos["video_id"])

print(f"Comentarios totales:                          {len(comentarios)}")
print(f"Comentarios asociados a un video del catálogo: {comentarios_con_video.sum()} "
      f"({comentarios_con_video.mean():.1%})")
print(f"Comentarios sin video correspondiente:         {(~comentarios_con_video).sum()}")
print()
videos_con_comentarios = videos["video_id"].isin(comentarios["video_id"])
print(f"Videos totales:                                {len(videos)}")
print(f"Videos con al menos un comentario:             {videos_con_comentarios.sum()} "
      f"({videos_con_comentarios.mean():.1%})")
print(f"Videos sin ningún comentario recolectado:      {(~videos_con_comentarios).sum()}")

Comentarios totales:                          406
Comentarios asociados a un video del catálogo: 406 (100.0%)
Comentarios sin video correspondiente:         0

Videos totales:                                293
Videos con al menos un comentario:             19 (6.5%)
Videos sin ningún comentario recolectado:      274


In [11]:
# Comentarios por video, únicamente entre los videos que sí tienen comentarios
por_video = comentarios.groupby("video_id").size().sort_values(ascending=False)
print("Videos con comentarios:", len(por_video))
print("Comentarios por video -> mínimo:", por_video.min(),
      "| mediana:", por_video.median(),
      "| máximo:", por_video.max())
por_video.head(10).to_frame("n_comentarios")

Videos con comentarios: 19
Comentarios por video -> mínimo: 1 | mediana: 7.0 | máximo: 161


,n_comentarios
video_id,
n8iP75gIpmw,161
j43HgwYFKfk,50
6W4u8sGEnGM,45
lj983NWyAQY,25
PjmxCj-a9Hg,25
OkXlHx0hx-8,25
yLZS3JiEBg8,16
06mFNPU0aB8,14
ndAZjHqzzT8,12


**Resultado de la integración**

Los **406 comentarios (100%) pudieron asociarse con un video** del catálogo mediante `video_id`; ninguno quedó huérfano.

La relación en el sentido inverso es muy distinta. **Solo 19 de los 293 videos (6.5%) tienen comentarios recolectados**, y 274 videos no tienen ninguno. Además, el reparto entre esos 19 videos es muy desigual, la mediana es de 7 comentarios por video, pero un solo video concentra 161.

El catálogo de videos y la muestra de comentarios no cubren lo mismo, así que cualquier red construida sobre los comentarios describirá únicamente a esos 19 videos. La ausencia de comentarios en los otros 274 **no significa que no los tengan**, sino que no fueron recolectados.